# Phase 3 — Knowledge-Graph-Based Zero-Hallucination Guardrails

Runs the exact 5-step algorithm from the methodology: generate a candidate response, extract triples from context and response, check structural alignment, verify or trigger refinement.

**Part 1** runs the self-tests (E1-E5) — no Phase 1/2 model needed, fake data only, just to confirm the algorithm works.

**Part 2** (optional) wires the guardrail onto your REAL Phase 2 `TheologicalAgent`, so real answers get real verification.


## 1. Upload the code

In [ ]:
from google.colab import files

print("Select Phase3_Guardrail_Code.zip")
uploaded = files.upload()


Select Phase3_Guardrail_Code.zip


Saving Phase3_Guardrail_Code.zip to Phase3_Guardrail_Code (1).zip


In [ ]:
!unzip -o -q Phase3_Guardrail_Code.zip -d Phase3
%cd Phase3
!find . -type f


/content/Phase3
./b5_real_finetuned/checkpoint-388/config.json
./b5_real_finetuned/checkpoint-388/config_sentence_transformers.json
./b5_real_finetuned/checkpoint-388/model.safetensors
./Phase3_Guardrail_Code/requirements.txt
./Phase3_Guardrail_Code/README.md
./Phase3_Guardrail_Code/src/e1_triple_extraction.py
./Phase3_Guardrail_Code/src/e4_zero_hallucination_guardrail.py
./Phase3_Guardrail_Code/src/e5_phase2_integration.py
./Phase3_Guardrail_Code/src/e2_knowledge_graph.py
./Phase3_Guardrail_Code/src/e3_structural_alignment.py


## 2. Install dependencies

In [ ]:
!pip install -q networkx rapidfuzz pyarabic


In [ ]:
import os
print("Current directory:", os.getcwd())
print("Contents:", os.listdir("."))
print("\nDoes Phase3/ exist?", os.path.exists("Phase3"))
if os.path.exists("Phase3"):
    print("Contents of Phase3/:", os.listdir("Phase3"))
    print("Contents of Phase3/src/:", os.listdir("Phase3/src") if os.path.exists("Phase3/src") else "src/ missing")

Current directory: /content/Phase3
Contents: ['b5_real_finetuned', 'Phase3_Guardrail_Code']

Does Phase3/ exist? False


In [ ]:
%cd Phase3_Guardrail_Code

/content/Phase3/Phase3_Guardrail_Code


## 3. Run the self-tests (Part 1 — no Phase 1/2 needed)

Each script proves itself with fake data. Run in order — later ones depend on earlier ones being correct.

### E1 — Triple extraction

In [ ]:
!python src/e1_triple_extraction.py


[TEST 1] Extracting from a verb-anchored sentence...
  Input: إن الله غفور رحيم
  -> (إن الله غفور رحيم | mentions | إن الله غفور رحيم)
[PASS]

[TEST 2] Extracting from a real tafsir-style sentence with a known verb...
  Input: خلق الله السماوات والأرض في ستة أيام
  -> ((implicit: Allah) | خلق | الله السماوات والأرض في ستة أيام) (source: 7:54)
[PASS]

[TEST 3] Extracting from context list (Phase 2 format)...
  Extracted 2 triples from 2 context items:
  -> (بسم الله الرحمن الرحيم | mentions | بسم الله الرحمن الرحيم) (source: 1:1)
  -> (أيوب | صبر | على البلاء وشكر الله) (source: 21:83)
[PASS]

[TEST 4] Fallback strategy on a clause with no known verb/preposition...
  Input: الحمد والثناء
  -> (الحمد والثناء | mentions | الحمد والثناء)
[PASS]

[RESULT] All E1 self-tests passed.


In [ ]:
!ls Phase3/src/

ls: cannot access 'Phase3/src/': No such file or directory


### E2 — Knowledge graph construction

In [ ]:
!python src/e2_knowledge_graph.py


[TEST 1] Building a graph from a small triple set...
Test graph: 2 nodes, 1 edges
  (implicit: Allah) --[خلق]--> الله السماوات والأرض
[PASS]

[TEST 2] Building Graph_C from a realistic context list...
Graph_C: 4 nodes, 3 edges
  بسم الله الرحمن الرحيم --[mentions]--> بسم الله الرحمن الرحيم
  أيوب --[صبر]--> على البلاء
  الله لا اله الا هو الحي القيوم --[mentions]--> الله لا اله الا هو الحي القيوم
[PASS]

[TEST 3] Building Graph_R from a candidate LLM response...
Graph_R: 3 nodes, 2 edges
  (implicit: Allah) --[خلق]--> الله الانسان من طين
  وأمر الله الملائكة بالسجود لادم --[mentions]--> وأمر الله الملائكة بالسجود لادم
[PASS]

[RESULT] All E2 self-tests passed.


### E3 — Structural alignment check

In [ ]:
!python src/e3_structural_alignment.py


[TEST 1] Fully aligned response (claims match context)...

Mismatch score: 0.000 (FULLY ALIGNED)
Aligned: 1/1
  [OK] (sim=100.0) ('أيوب', 'صبر', 'على البلاء')
[PASS]

[TEST 2] Hallucinated response (claims NOT in context)...

Mismatch score: 1.000 (MISMATCH DETECTED)
Aligned: 0/1
  [MISMATCH] (sim=3.2) ('موسى بنى الكعبة', 'في', 'مكة')
      closest context match: ('أيوب', 'صبر', 'على البلاء')
[PASS]

[RESULT] All E3 self-tests passed.


### E4 — Full guardrail algorithm

In [ ]:
!python src/e4_zero_hallucination_guardrail.py


[TEST 1] A grounded response should verify on the first attempt...

[GUARDRAIL ATTEMPT 1] Query: 'من صبر على البلاء'

Mismatch score: 0.000 (FULLY ALIGNED)
Aligned: 1/1
  [OK] (sim=100.0) ('أيوب', 'صبر', 'على البلاء')
[GUARDRAIL] Mismatch score = 0 -> response VERIFIED.

############################################################
Query: من صبر على البلاء
Verified: True (stopped: verified)
Attempts used: 1
Final mismatch score: 0.0

VERIFIED RESPONSE:
أيوب صبر على البلاء
############################################################

[PASS] Test 1 passed.

[TEST 2] A hallucinating LLM should get rejected after exhausting refinement attempts...

[GUARDRAIL ATTEMPT 1] Query: 'من صبر على البلاء'

Mismatch score: 1.000 (MISMATCH DETECTED)
Aligned: 0/1
  [MISMATCH] (sim=5.8) ('موسى بنى الكعبة', 'في', 'مكة')
      closest context match: ('أيوب', 'صبر', 'على البلاء وشكر الله')
[GUARDRAIL] Mismatch score = 1.000 -> triggering refinement (attempt 1/2).

[GUARDRAIL ATTEMPT 2] Query: 'من صبر على ال

### E5 — Phase 2 integration wiring

In [ ]:
!python src/e5_phase2_integration.py


[TEST 1] Phase 2 gives a grounded answer immediately...

[GUARDRAIL ATTEMPT 1] Query: 'من صبر على البلاء'

Mismatch score: 0.000 (FULLY ALIGNED)
Aligned: 1/1
  [OK] (sim=100.0) ('أيوب', 'صبر', 'على البلاء')
[GUARDRAIL] Mismatch score = 0 -> response VERIFIED.

############################################################
Query: من صبر على البلاء
Verified: True (stopped: verified)
Attempts used: 1
Final mismatch score: 0.0

VERIFIED RESPONSE:
أيوب صبر على البلاء
############################################################

[PASS] Test 1 passed.

[TEST 2] Phase 2 hallucinates first, then gives a grounded answer on refinement...

[GUARDRAIL ATTEMPT 1] Query: 'من صبر على البلاء'

Mismatch score: 1.000 (MISMATCH DETECTED)
Aligned: 0/1
  [MISMATCH] (sim=14.2) ('فرعون طغى', 'في', 'الارض')
      closest context match: ('أيوب', 'صبر', 'على البلاء وشكر الله')
[GUARDRAIL] Mismatch score = 1.000 -> triggering refinement (attempt 1/3).

[GUARDRAIL ATTEMPT 2] Query: 'من صبر على البلاء (refined)'

Mis

All five should end with `[PASS]`/`[RESULT] ... passed` lines. If anything fails, paste the error back before moving to Part 2.


---
## Part 2 (optional) — Run against your REAL Phase 1 + Phase 2 pipeline

Needs: Phase 1's model/index, and Phase 2's D1/D2/D3 code + Roma's C1/C2/C3 outputs (the same Sync Point assets from your Phase 2 integration notebook).

## 4. Mount Drive and locate everything

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PHASE1_DIR = "/content/drive/MyDrive/Phase1_Project/MemberB_B4_B6_output"
ROMA_DIR = "/content/drive/MyDrive/Phase2_Project/Roma_output"

print("Phase 1:")
!ls "{PHASE1_DIR}"

print("Roma's Phase 2:")
!ls "{ROMA_DIR}"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Phase 1:
b5_real_finetuned  data  index	outputs
Roma's Phase 2:
data  quranNLP	src


## 5. Upload your Phase 2 D1/D2/D3 code (Laiba's track)

In [ ]:
print("Select d1_agent_loop.py, d2_cot_prompting.py, d3_sufficiency_and_fallback.py together")
uploaded_d = files.upload()

import shutil
for filename in list(uploaded_d.keys()):
    shutil.move(filename, f"src/{filename}")
print("Placed:", [f for f in uploaded_d.keys()])


Select d1_agent_loop.py, d2_cot_prompting.py, d3_sufficiency_and_fallback.py together


Saving d1_agent_loop.py to d1_agent_loop.py
Saving d2_cot_prompting.py to d2_cot_prompting.py
Saving d3_sufficiency_and_fallback.py to d3_sufficiency_and_fallback.py
Placed: ['d1_agent_loop.py', 'd2_cot_prompting.py', 'd3_sufficiency_and_fallback.py']


## 6. Copy Phase 1's model/index and Roma's C1/C2/C3 outputs

In [ ]:
import shutil, os

shutil.copytree(f"{PHASE1_DIR}/b5_real_finetuned", "b5_real_finetuned", dirs_exist_ok=True)
shutil.copytree(f"{PHASE1_DIR}/index", "index", dirs_exist_ok=True)

if not os.path.exists("src/b6_build_index_and_retrieval_api.py"):
    shutil.copy(f"{ROMA_DIR}/src/b6_build_index_and_retrieval_api.py", "src/b6_build_index_and_retrieval_api.py")

os.makedirs("quranNLP/shared/data", exist_ok=True)
os.makedirs("data", exist_ok=True)
shutil.copy(f"{ROMA_DIR}/quranNLP/shared/data/ayatec_records.json", "quranNLP/shared/data/ayatec_records.json")
shutil.copy(f"{ROMA_DIR}/quranNLP/shared/data/squad_v2_sample.json", "quranNLP/shared/data/squad_v2_sample.json")
shutil.copy(f"{ROMA_DIR}/data/sufficiency_labels.json", "data/sufficiency_labels.json")
shutil.copy(f"{ROMA_DIR}/src/c1_cross_lingual_fallback.py", "src/c1_cross_lingual_fallback.py")
shutil.copy(f"{ROMA_DIR}/src/c3_query_refinement.py", "src/c3_query_refinement.py")

print("All Phase 1 + Phase 2 assets in place.")


All Phase 1 + Phase 2 assets in place.


## 7. Set up your LLM (same pattern as Phase 2 — Groq via Colab Secrets)

In [ ]:
!pip install -q sentence-transformers hnswlib pandas groq

import time
from groq import Groq
from google.colab import userdata

groq_client = Groq(api_key=userdata.get('GROQ_API_KEY'))

def call_llm_with_retry(prompt: str, max_retries: int = 3) -> str:
    for attempt in range(max_retries):
        try:
            response = groq_client.chat.completions.create(
                model="llama-3.3-70b-versatile",
                messages=[{"role": "user", "content": prompt}],
                temperature=0,
                max_tokens=1000,
            )
            return response.choices[0].message.content
        except Exception as e:
            if "rate_limit" in str(e).lower() and attempt < max_retries - 1:
                wait_time = 15 * (attempt + 1)
                print(f"[RATE LIMIT] Waiting {wait_time}s...")
                time.sleep(wait_time)
            else:
                raise

def call_llm(prompt: str) -> str:
    return call_llm_with_retry(prompt)

print("LLM ready.")


## 8. Build the real, complete TheologicalAgent (Phase 2)

In [ ]:
import sys
sys.path.insert(0, "src")
import json

from sentence_transformers import SentenceTransformer
from b6_build_index_and_retrieval_api import load_index, RetrievalAPI

model = SentenceTransformer("./b5_real_finetuned")
index, entries = load_index(dim=model.get_sentence_embedding_dimension(), out_dir="index")
retrieval_api = RetrievalAPI(model, index, entries)
print(f"Retrieval API ready: {len(entries)} indexed entries.")

from c1_cross_lingual_fallback import CrossLingualLookup
from d3_sufficiency_and_fallback import TheologicalAgent, SufficiencyScorer, CrossLingualFallback
from d1_agent_loop import AgentConfig
from c3_query_refinement import refine_query

with open("quranNLP/shared/data/ayatec_records.json", encoding="utf-8") as f:
    ayatec_records = json.load(f)
with open("quranNLP/shared/data/squad_v2_sample.json", encoding="utf-8") as f:
    squad_records = json.load(f)
lookup = CrossLingualLookup(ayatec_records, squad_records)
fallback = CrossLingualFallback(lookup_fn=lookup)

with open("data/sufficiency_labels.json", encoding="utf-8") as f:
    labeled_examples = json.load(f)
scorer = SufficiencyScorer()
scorer.calibrate(labeled_examples)

theological_agent = TheologicalAgent(
    retrieval_api=retrieval_api,
    call_llm_fn=call_llm,
    sufficiency_scorer=scorer,
    fallback=fallback,
    refine_fn=refine_query,
    config=AgentConfig(max_iterations=3, verbose=False),
    max_chars_per_context_item=400,
)
print("Phase 2 TheologicalAgent fully assembled.")


## 9. Run real queries through the Phase 3 guardrail

This is the full pipeline: Phase 1 retrieval + Phase 2 agentic CoT reasoning + Phase 3 structural verification.

In [ ]:
from e5_phase2_integration import verify_theological_agent_answer
from e4_zero_hallucination_guardrail import GuardrailConfig, print_guardrail_result

test_queries = [
    "ما فوائد الصبر في القرآن",
    "من هو النبي المعروف بالصبر",
    "ماذا يقول القرآن عن الرحمة",
    "التوبة والاستغفار",
    "العدل في الإسلام",
    "الشكر لله",
    "الخوف من الله",
    "الايمان بالغيب",
    "الصدق في القول",
    "بر الوالدين",
]

guardrail_results = []
for q in test_queries:
    print(f"\n{'='*70}\nQUERY: {q}\n{'='*70}")
    result = verify_theological_agent_answer(
        theological_agent, q,
        refine_query_fn=refine_query,
        guardrail_config=GuardrailConfig(similarity_threshold=65.0, max_refinement_attempts=2, verbose=True),
    )
    print_guardrail_result(result)
    guardrail_results.append({
        "query": q,
        "verified": result.verified,
        "stopped_reason": result.stopped_reason,
        "attempts": len(result.attempts),
        "final_mismatch_score": result.final_mismatch_score,
        "final_response": result.final_response,
    })


## 10. Save results to Drive

In [ ]:
import json

DEST = "/content/drive/MyDrive/Phase3_Project/guardrail_output"
!mkdir -p "{DEST}"

with open("phase3_guardrail_results.json", "w", encoding="utf-8") as f:
    json.dump(guardrail_results, f, ensure_ascii=False, indent=2)

!cp phase3_guardrail_results.json "{DEST}/"
!cp -r src "{DEST}/"

print(f"Saved to: {DEST}")
!find "{DEST}" -maxdepth 2


## 11. Where this leaves the project

If the results above show queries getting verified (or correctly rejected when the model can't produce a grounded answer), Phase 3's hallucination guardrail is functionally complete and wired onto the full Phase 1 + Phase 2 pipeline. This is the point to loop in Sir Atif with real, structurally-verified results — the "zero-hallucination" claim from the methodology now has actual enforcement behind it, not just a design description.